In [ ]:
# Multivariate Bayes classifier
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [30]:
def log_multivariate_gaussian_pdf(x, mean, inv_cov, logdet):
    # this equation P(X|Wk) = (1/(2*pi)^n/2 * |Ck|^0.5) * e^-0.5 (X-mean_k).TCk^-1(X-mean_k)
    # ln(P(X|wk)) = -0.5n * ln(2pi) -0.5*ln(|Ck|) -0.5 (X-mean_k).T * Ck^-1 * (X-mean_k)
    # the term -0.5n * ln(2pi) is constant and does not depend on classes we can ignore since we compare ln(P) nothing more less
    # n = len(mean), number of features
    x_mu = x - mean
    quad_form = x_mu.T @ inv_cov @ x_mu
    return -0.5 * logdet - 0.5 * quad_form #- 0.5 * len(mean) * np.log(2 * np.pi)

def estimate_parameters(X, y):
    params = {}
    n_samples = len(X)
    D = X.shape[1]  # Number of features

    for c in np.unique(y):
        X_c = X[y == c]
        mean = np.mean(X_c, axis=0)
        cov = np.cov(X_c, rowvar=False)
        cov += 0.1 * np.eye(D) # Add regularization to avoid singular matrix

        inv_cov = np.linalg.inv(cov)
        sign, logdet = np.linalg.slogdet(cov)
        # With regularization, sign should be +1; if not, handle edge case
        if sign <= 0:
            logdet = -np.inf  # Effectively zero probability
        prior = len(X_c) / float(n_samples)
        params[c] = { 
            "mean": mean, 
            "inv_cov": inv_cov,
            "logdet": logdet,
            "prior": prior}
    return params


def predict_mvnb(X, params):
    preds = []
    for x in X:
        scores = {}
        for c in params:
            mean = params[c]["mean"]
            inv_cov = params[c]["inv_cov"]
            logdet = params[c]["logdet"]
            prior = params[c]["prior"]
            scores[c] = np.log(prior) +  log_multivariate_gaussian_pdf(x, mean, inv_cov, logdet)
        
        preds.append(max(scores, key=scores.get))
    return np.array(preds)


In [31]:
# Load dataset
X, y = fetch_openml(name='mnist_784',version=1, return_X_y=True, cache=True)
# convert to numpy array
X = np.array(X) / 255.0
y = np.array(y).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=1)

In [32]:
par = estimate_parameters(X_train, y_train)
pred = predict_mvnb(X_test, par)

accuracy = np.mean(pred == y_test)
print(f'accuracy: {accuracy}')
print(classification_report(y_test, pred))

accuracy: 0.9492857142857143
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      1380
           1       0.91      0.99      0.95      1632
           2       0.97      0.94      0.95      1433
           3       0.96      0.93      0.94      1431
           4       0.98      0.95      0.96      1328
           5       0.96      0.93      0.95      1297
           6       0.97      0.97      0.97      1331
           7       0.96      0.95      0.95      1444
           8       0.94      0.89      0.91      1351
           9       0.90      0.95      0.92      1373

    accuracy                           0.95     14000
   macro avg       0.95      0.95      0.95     14000
weighted avg       0.95      0.95      0.95     14000

